In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/nirbaan_project"

folders = [
    "data",
    "data/fl_splits",
    "data/fl_splits/noniid",
    "notebooks",
    "notebooks/scripts",
    "configs",
    "outputs",
    "outputs/centralized_baseline",
    "outputs/federated_baseline",
    "artifacts",
    "artifacts/centralized_adapter",
    "artifacts/federated_round_adapters",
    "artifacts/gguf_model",
    "artifacts/federated_final_gguf",
    "eval",
    "paper_assets",
    "paper_assets/tables",
    "paper_assets/examples",
    "paper_assets/plots",
]

for folder in folders:
    os.makedirs(os.path.join(BASE_PATH, folder), exist_ok=True)

print("Created project structure at:", BASE_PATH)

Created project structure at: /content/drive/MyDrive/nirbaan_project


In [ ]:
!pip -q install --upgrade pip
!pip -q uninstall -y flwr unsloth unsloth_zoo transformers trl peft bitsandbytes accelerate protobuf grpcio grpcio-status typer typer-slim
!pip -q install "flwr[simulation]==1.13.0"
!pip -q install "unsloth[colab-new]"
!pip -q install datasets sentencepiece pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 49.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-pubsub 2.36.0 requires grpcio-status>=1.33.2, which is not installed.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, which is not installed.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.8 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.8 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.8 which is incompatible.


In [ ]:
!pip uninstall -y flwr ray
!pip install -U "flwr[simulation]"

Found existing installation: flwr 1.13.0
Uninstalling flwr-1.13.0:
  Successfully uninstalled flwr-1.13.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 28.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.4/71.4 MB 83.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 83.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 88.0 MB/s  0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.8
    Uninstalling protobuf-4.25.8:
      Successfully uninstalled protobuf-4.25.8
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.64.3
    Uninstalling grpcio-1.64.3:
      Successfully uninstalled grpcio-1.64.3
  Attempting uninstall: cryptography
    Found existing installation: cryptography 42.0.8
    Uninstalling cryptography-42.0.8:
      Successfully uninstalled cryptography-42.0.8
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [flwr]
ERROR: pip's dependency resolver does no

In [ ]:
import flwr, ray
print("flwr:", flwr.__version__)
print("ray:", ray.__version__)

flwr: 1.27.0
ray: 2.51.1


In [ ]:
import torch
import flwr
import transformers
import trl
import peft
import datasets
import bitsandbytes
import unsloth

print("torch:", torch.__version__)
print("flwr:", flwr.__version__)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("datasets:", datasets.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("unsloth:", unsloth.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/tmp/ipykernel_473/2018765010.py:8: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import unsloth


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
torch: 2.10.0+cu128
flwr: 1.27.0
transformers: 5.3.0
trl: 0.24.0
peft: 0.18.1
datasets: 4.3.0
bitsandbytes: 0.49.2
unsloth: 2026.3.8
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


In [ ]:
import os, json, torch, flwr, transformers, trl, peft, datasets, bitsandbytes, unsloth

BASE_PATH = "/content/drive/MyDrive/nirbaan_project"
os.makedirs(os.path.join(BASE_PATH, "configs"), exist_ok=True)

versions = {
    "torch": torch.__version__,
    "flwr": flwr.__version__,
    "transformers": transformers.__version__,
    "trl": trl.__version__,
    "peft": peft.__version__,
    "datasets": datasets.__version__,
    "bitsandbytes": bitsandbytes.__version__,
    "unsloth": unsloth.__version__,
}

with open(os.path.join(BASE_PATH, "configs", "package_versions.json"), "w") as f:
    json.dump(versions, f, indent=2)

print("Saved package versions.")

Saved package versions.


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
BF16 supported: True


In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))

Saving dev.jsonl to dev.jsonl
Saving test.jsonl to test.jsonl
Saving train.jsonl to train.jsonl
['dev.jsonl', 'test.jsonl', 'train.jsonl']


In [ ]:
import shutil

for fname in ["train.jsonl", "dev.jsonl", "test.jsonl"]:
    shutil.copy(fname, os.path.join(BASE_PATH, "data", fname))

print("Copied dataset to Drive.")

Copied dataset to Drive.


In [ ]:
TRAIN_PATH = os.path.join(BASE_PATH, "data", "train.jsonl")
DEV_PATH   = os.path.join(BASE_PATH, "data", "dev.jsonl")
TEST_PATH  = os.path.join(BASE_PATH, "data", "test.jsonl")
SPLIT_DIR  = os.path.join(BASE_PATH, "data", "fl_splits", "noniid")

In [ ]:
import json
from collections import defaultdict, Counter

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception as e:
                raise ValueError(f"Bad JSON in {path}, line {line_num}: {e}")
    return rows

def save_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def append_jsonl(path, row):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

In [ ]:
train_records = load_jsonl(TRAIN_PATH)
print("Train records:", len(train_records))
print(json.dumps(train_records[0], indent=2, ensure_ascii=False)[:1500])

Train records: 797
{
  "instruction": "Act as an ERP therapist. Generate an imaginal exposure script based on the following OCD obsession, compulsion, and feared consequence.",
  "input": "Obsession: I could smother my baby to death with a pillow. Compulsion: Avoiding being alone with the baby; checking on the baby repeatedly; confessing thoughts to spouse and seeking reassurance. Feared consequence: Actually smothering my baby; being arrested for murder; losing my marriage and going to prison. Script intensity: 10/10 Exposure type: imaginal",
  "output": "I am home alone with my newborn daughter Lily, who is sleeping peacefully in her crib. As I watch her, a horrifying thought enters my mind: 'I could smother her with a pillow.' My heart races and my palms sweat as the image of me placing a pillow over her face and suffocating her consumes my thoughts. What if I actually do it? What if I'm capable of such evil? I feel like I'm losing my mind. I start to pace the room, my breath coming

In [ ]:
import random
import os
import json
from collections import defaultdict, Counter

NUM_CLIENTS = 4
SEED = 42
random.seed(SEED)

# Choose 3 dominant themes for 3 clients
dominant_types = [
    "harm ocd",
    "contamination ocd",
    "checking/hit-and-run ocd",
]

train_records = load_jsonl(TRAIN_PATH)

missing_type = [i for i, r in enumerate(train_records) if "type" not in r]
if missing_type:
    raise ValueError(f"{len(missing_type)} training rows are missing the 'type' field")

def normalize_type(type_name):
    return str(type_name).strip().lower()

# Group records by OCD type
by_type = defaultdict(list)
for rec in train_records:
    ocd_type = normalize_type(rec["type"])
    by_type[ocd_type].append(rec)

# Shuffle records inside each type
for ocd_type in by_type:
    random.shuffle(by_type[ocd_type])

clients = [[] for _ in range(NUM_CLIENTS)]
mixed_pool = []

# Client 0 -> mostly Harm OCD
# Client 1 -> mostly Contamination OCD
# Client 2 -> mostly Checking/Hit-and-Run OCD
for client_id, ocd_type in enumerate(dominant_types):
    type_rows = by_type.get(ocd_type, [])
    cut = int(0.8 * len(type_rows))  # 80% stays dominant in that client
    clients[client_id].extend(type_rows[:cut])
    mixed_pool.extend(type_rows[cut:])  # remaining 20% goes to shared pool

# All other OCD types go to mixed pool
for ocd_type, rows in by_type.items():
    if ocd_type not in dominant_types:
        mixed_pool.extend(rows)

# Shuffle shared pool
random.shuffle(mixed_pool)

# Distribute mixed pool across all clients
for idx, rec in enumerate(mixed_pool):
    clients[idx % NUM_CLIENTS].append(rec)

# Save client files and manifest
manifest = {
    "num_clients": NUM_CLIENTS,
    "seed": SEED,
    "split_type": "noniid",
    "dominant_types": dominant_types,
    "clients": []
}

for cid in range(NUM_CLIENTS):
    random.shuffle(clients[cid])  # final shuffle

    client_path = os.path.join(SPLIT_DIR, f"client_{cid}.jsonl")
    write_jsonl(client_path, clients[cid])

    type_counts = Counter([normalize_type(r["type"]) for r in clients[cid]])

    manifest["clients"].append({
        "client_id": cid,
        "path": client_path,
        "num_samples": len(clients[cid]),
        "type_counts": dict(sorted(type_counts.items())),
    })

manifest_path = os.path.join(SPLIT_DIR, "split_manifest.json")
save_json(manifest_path, manifest)

print(json.dumps(manifest, indent=2, ensure_ascii=False))

{
  "num_clients": 4,
  "seed": 42,
  "split_type": "noniid",
  "dominant_types": [
    "harm ocd",
    "contamination ocd",
    "checking/hit-and-run ocd"
  ],
  "clients": [
    {
      "client_id": 0,
      "path": "/content/drive/MyDrive/nirbaan_project/data/fl_splits/noniid/client_0.jsonl",
      "num_samples": 223,
      "type_counts": {
        "checking/hit-and-run ocd": 4,
        "contamination ocd": 3,
        "existential/schizophrenia ocd": 19,
        "harm ocd": 83,
        "pedophilia ocd (pocd)": 19,
        "postpartum ocd": 24,
        "relationship ocd (rocd)": 18,
        "scrupulosity ocd": 23,
        "sexual orientation ocd (hocd)": 30
      }
    },
    {
      "client_id": 1,
      "path": "/content/drive/MyDrive/nirbaan_project/data/fl_splits/noniid/client_1.jsonl",
      "num_samples": 211,
      "type_counts": {
        "checking/hit-and-run ocd": 4,
        "contamination ocd": 68,
        "existential/schizophrenia ocd": 19,
        "harm ocd": 2,
       

In [ ]:
split_stats_path = os.path.join(BASE_PATH, "paper_assets", "tables", "split_stats.json")
save_json(split_stats_path, manifest)
print("Saved split stats to:", split_stats_path)

Saved split stats to: /content/drive/MyDrive/nirbaan_project/paper_assets/tables/split_stats.json


In [ ]:
MODEL_NAME = "mlabonne/Meta-Llama-3.1-8B-Instruct-abliterated"
MAX_SEQ_LENGTH = 2048

FL_CONFIG = {
    "experiment_name": "federated_qlora_flower_baseline",
    "base_model": MODEL_NAME,
    "num_clients": NUM_CLIENTS,
    "num_rounds": 5,
    "local_epochs": 1,
    "learning_rate": 2e-4,
    "per_device_train_batch_size": 4,
    "per_device_eval_batch_size": 4,
    "gradient_accumulation_steps": 2,
    "lora_r": 16,
    "lora_alpha": 16,
    "lora_dropout": 0.0,
    "target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    "seed": 42,
    "quantized_base_model": True,
    "client_split_manifest": manifest_path,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
}

FL_CONFIG_PATH = os.path.join(BASE_PATH, "configs", "federated_run_config.json")
save_json(FL_CONFIG_PATH, FL_CONFIG)
print("Saved config to:", FL_CONFIG_PATH)

Saved config to: /content/drive/MyDrive/nirbaan_project/configs/federated_run_config.json


In [ ]:
from datasets import Dataset

def normalize_record(rec):
    instruction = str(rec.get("instruction", "")).strip()
    user_input = str(rec.get("input", "")).strip()
    response = str(rec.get("output", "")).strip()
    ocd_type = str(rec.get("type", "")).strip()

    if instruction and user_input:
        prompt = f"{instruction}\n\n{user_input}"
    else:
        prompt = instruction or user_input

    return {
        "prompt": prompt,
        "response": response,
        "type": ocd_type,
    }

def build_hf_dataset(jsonl_path):
    raw = load_jsonl(jsonl_path)
    rows = []

    for rec in raw:
        item = normalize_record(rec)

        if (
            item["prompt"]
            and item["response"]
            and len(item["response"]) >= 30
        ):
            rows.append(item)

    return Dataset.from_list(rows)

def to_chat(example):
    return {
        "messages": [
            {"role": "user", "content": example["prompt"]},
            {"role": "assistant", "content": example["response"]},
        ],
        "type": example["type"],
    }

def apply_template(ds, tokenizer):
    ds = ds.map(to_chat)

    def _apply(batch):
        texts = [
            tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
            for messages in batch["messages"]
        ]
        return {"text": texts}

    ds = ds.map(_apply, batched=True, batch_size=32)
    return ds

In [ ]:
import numpy as np
import torch
from collections import OrderedDict
from unsloth import FastLanguageModel

def build_model_and_tokenizer():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=FL_CONFIG["lora_r"],
        target_modules=FL_CONFIG["target_modules"],
        lora_alpha=FL_CONFIG["lora_alpha"],
        lora_dropout=FL_CONFIG["lora_dropout"],
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=FL_CONFIG["seed"],
    )
    return model, tokenizer

def get_lora_state_dict(model):
    state = model.state_dict()
    lora_state = OrderedDict()
    for k in sorted(state.keys()):
        if "lora_" in k:
            lora_state[k] = state[k].detach().cpu()
    return lora_state

def get_lora_ndarrays(model):
    lora_state = get_lora_state_dict(model)
    return [v.numpy() for v in lora_state.values()]

def set_lora_ndarrays(model, parameters):
    lora_state = get_lora_state_dict(model)
    keys = list(lora_state.keys())

    if len(keys) != len(parameters):
        raise ValueError(f"Mismatch: expected {len(keys)} tensors, got {len(parameters)}")

    new_state = {}
    for k, arr in zip(keys, parameters):
        ref = lora_state[k]
        new_state[k] = torch.tensor(arr, dtype=ref.dtype)

    model.load_state_dict(new_state, strict=False)

def cleanup_memory(*objs):
    for obj in objs:
        del obj
    torch.cuda.empty_cache()

In [ ]:
import shutil
import tempfile
import torch
from trl import SFTTrainer, SFTConfig

def train_one_client(client_jsonl_path, global_lora_params, round_config, client_id, server_round):
    model, tokenizer = build_model_and_tokenizer()
    set_lora_ndarrays(model, global_lora_params)

    train_ds = build_hf_dataset(client_jsonl_path)
    train_ds = apply_template(train_ds, tokenizer)

    tmp_out = tempfile.mkdtemp()

    training_args = SFTConfig(
        output_dir=tmp_out,
        per_device_train_batch_size=int(round_config["per_device_train_batch_size"]),
        gradient_accumulation_steps=int(round_config["gradient_accumulation_steps"]),
        num_train_epochs=int(round_config["local_epochs"]),
        learning_rate=float(round_config["learning_rate"]),
        logging_steps=10,
        save_strategy="no",
        eval_strategy="no",
        report_to="none",
        optim="paged_adamw_8bit",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        dataloader_pin_memory=False,
        seed=FL_CONFIG["seed"] + client_id + server_round,
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        processing_class=tokenizer,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,
    )

    result = trainer.train()
    updated_params = get_lora_ndarrays(model)

    metrics = {
        "train_loss": float(result.training_loss),
        "num_examples": len(train_ds),
        "client_id": int(client_id),
        "server_round": int(server_round),
    }

    cleanup_memory(trainer, model, tokenizer)
    shutil.rmtree(tmp_out, ignore_errors=True)

    return updated_params, len(train_ds), metrics

In [ ]:
import os
import shutil
import tempfile
import torch
from trl import SFTTrainer, SFTConfig

DEV_METRICS_LOG = os.path.join(BASE_PATH, "outputs", "federated_baseline", "server_eval.jsonl")
CLIENT_FIT_LOG  = os.path.join(BASE_PATH, "outputs", "federated_baseline", "round_logs.jsonl")
ROUND_ADAPTERS_DIR = os.path.join(BASE_PATH, "artifacts", "federated_round_adapters")
BEST_ADAPTER_DIR   = os.path.join(BASE_PATH, "artifacts", "federated_best_adapter")

os.makedirs(os.path.dirname(DEV_METRICS_LOG), exist_ok=True)
os.makedirs(ROUND_ADAPTERS_DIR, exist_ok=True)

best_tracker = {"best_loss": float("inf"), "best_round": -1}


def save_adapter_from_params(parameters, out_dir):
    os.makedirs(out_dir, exist_ok=True)

    model, tokenizer = build_model_and_tokenizer()
    set_lora_ndarrays(model, parameters)

    model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)

    cleanup_memory(model, tokenizer)


def evaluate_global_model(server_round, parameters, config):
    model, tokenizer = build_model_and_tokenizer()
    set_lora_ndarrays(model, parameters)

    dev_ds = build_hf_dataset(DEV_PATH)
    dev_ds = apply_template(dev_ds, tokenizer)

    tmp_out = tempfile.mkdtemp()

    eval_args = SFTConfig(
        output_dir=tmp_out,
        per_device_eval_batch_size=int(FL_CONFIG["per_device_eval_batch_size"]),
        do_train=False,
        do_eval=True,
        report_to="none",
        disable_tqdm=True,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        dataloader_pin_memory=False,
    )

    trainer = SFTTrainer(
        model=model,
        args=eval_args,
        train_dataset=dev_ds,   # workaround for current Unsloth init path
        eval_dataset=dev_ds,
        processing_class=tokenizer,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,
    )

    # Remove notebook-specific progress callback that crashes on eval-before-train
    try:
        from transformers.utils.notebook import NotebookProgressCallback
        trainer.remove_callback(NotebookProgressCallback)
    except Exception:
        pass

    metrics = trainer.evaluate()
    eval_loss = float(metrics["eval_loss"])

    round_adapter_dir = os.path.join(ROUND_ADAPTERS_DIR, f"round_{server_round}")
    save_adapter_from_params(parameters, round_adapter_dir)

    append_jsonl(
        DEV_METRICS_LOG,
        {
            "server_round": int(server_round),
            "eval_loss": eval_loss,
            "metrics": metrics,
            "round_adapter_dir": round_adapter_dir,
        },
    )

    if eval_loss < best_tracker["best_loss"]:
        if os.path.exists(BEST_ADAPTER_DIR):
            shutil.rmtree(BEST_ADAPTER_DIR)
        shutil.copytree(round_adapter_dir, BEST_ADAPTER_DIR)
        best_tracker["best_loss"] = eval_loss
        best_tracker["best_round"] = int(server_round)

    cleanup_memory(trainer, model, tokenizer)
    shutil.rmtree(tmp_out, ignore_errors=True)

    return eval_loss, {"eval_loss": eval_loss}

In [ ]:
import flwr as fl
from flwr.client import NumPyClient, ClientApp
from flwr.common import Context

class ERPFlowerClient(NumPyClient):
    def __init__(self, client_id, client_jsonl_path):
        self.client_id = client_id
        self.client_jsonl_path = client_jsonl_path

    def get_parameters(self, config):
        # Not heavily used once server initial_parameters are set
        model, tokenizer = build_model_and_tokenizer()
        params = get_lora_ndarrays(model)
        cleanup_memory(model, tokenizer)
        return params

    def fit(self, parameters, config):
        updated_params, num_examples, metrics = train_one_client(
            client_jsonl_path=self.client_jsonl_path,
            global_lora_params=parameters,
            round_config=config,
            client_id=self.client_id,
            server_round=int(config.get("server_round", 0)),
        )
        append_jsonl(CLIENT_FIT_LOG, metrics)
        return updated_params, num_examples, metrics

    def evaluate(self, parameters, config):
        return 0.0, 0, {}

def client_fn(context: Context):
    client_id = int(context.node_config.get("partition-id", 0))
    client_path = os.path.join(SPLIT_DIR, f"client_{client_id}.jsonl")
    return ERPFlowerClient(client_id, client_path).to_client()

client_app = ClientApp(client_fn=client_fn)

In [ ]:
import flwr as fl
from flwr.client import NumPyClient, ClientApp
from flwr.common import Context

class ERPFlowerClient(NumPyClient):
    def __init__(self, client_id, client_jsonl_path):
        self.client_id = client_id
        self.client_jsonl_path = client_jsonl_path

    def get_parameters(self, config):
        # Not heavily used once server initial_parameters are set
        model, tokenizer = build_model_and_tokenizer()
        params = get_lora_ndarrays(model)
        cleanup_memory(model, tokenizer)
        return params

    def fit(self, parameters, config):
        updated_params, num_examples, metrics = train_one_client(
            client_jsonl_path=self.client_jsonl_path,
            global_lora_params=parameters,
            round_config=config,
            client_id=self.client_id,
            server_round=int(config.get("server_round", 0)),
        )
        append_jsonl(CLIENT_FIT_LOG, metrics)
        return updated_params, num_examples, metrics

    def evaluate(self, parameters, config):
        return 0.0, 0, {}

def client_fn(context: Context):
    client_id = int(context.node_config.get("partition-id", 0))
    client_path = os.path.join(SPLIT_DIR, f"client_{client_id}.jsonl")
    return ERPFlowerClient(client_id, client_path).to_client()

client_app = ClientApp(client_fn=client_fn)

In [ ]:
from flwr.server import ServerApp, ServerConfig, ServerAppComponents
from flwr.server.strategy import FedAvg
from flwr.common import ndarrays_to_parameters

def fit_config(server_round: int):
    return {
        "server_round": server_round,
        "local_epochs": FL_CONFIG["local_epochs"],
        "learning_rate": FL_CONFIG["learning_rate"],
        "per_device_train_batch_size": FL_CONFIG["per_device_train_batch_size"],
        "gradient_accumulation_steps": FL_CONFIG["gradient_accumulation_steps"],
    }

def server_fn(context: Context):
    model, tokenizer = build_model_and_tokenizer()
    initial_params = ndarrays_to_parameters(get_lora_ndarrays(model))
    cleanup_memory(model, tokenizer)

    strategy = FedAvg(
        fraction_fit=1.0,
        fraction_evaluate=0.0,
        min_fit_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS,
        initial_parameters=initial_params,
        evaluate_fn=evaluate_global_model,
        on_fit_config_fn=fit_config,
        accept_failures=False,
    )

    config = ServerConfig(num_rounds=FL_CONFIG["num_rounds"])
    return ServerAppComponents(strategy=strategy, config=config)

server_app = ServerApp(server_fn=server_fn)

In [ ]:
from flwr.simulation import run_simulation

run_simulation(
    server_app=server_app,
    client_app=client_app,
    num_supernodes=NUM_CLIENTS,
    backend_config={
        "client_resources": {
            "num_cpus": 1,
            "num_gpus": 1.0,
        }
    },
)

DEBUG:flwr:backend_config: {'client_resources': {'num_cpus': 1, 'num_gpus': 1.0}}
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
DEBUG:flwr:Asyncio event loop already running.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes 

==((====))==  Unsloth 2026.3.8: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=473) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=473) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load mlabonne/Meta-Llama-3.1-8B-Instruct-abliterated as a legacy tokenizer.


mlabonne/Meta-Llama-3.1-8B-Instruct-abliterated does not have a padding token! Will use pad_token = <|finetune_right_pad_id|>.


INFO :      fit progress: (5, 0.8428990244865417, {'eval_loss': 0.8428990244865417}, 1946.27025442)
INFO :      configure_evaluate: no clients selected, skipping evaluation
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 5 round(s) in 1946.27s
INFO :      	History (loss, centralized):
INFO :      		round 0: 1.9746555089950562
INFO :      		round 1: 1.0983030796051025
INFO :      		round 2: 0.9791868329048157
INFO :      		round 3: 0.9002630710601807
INFO :      		round 4: 0.8599494695663452
INFO :      		round 5: 0.8428990244865417
INFO :      	History (metrics, centralized):
INFO :      	{'eval_loss': [(0, 1.9746555089950562),
INFO :      	               (1, 1.0983030796051025),
INFO :      	               (2, 0.9791868329048157),
INFO :      	               (3, 0.9002630710601807),
INFO :      	               (4, 0.8599494695663452),
INFO :      	               (5, 0.8428990244865417)]}
INFO :      
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: De

In [ ]:
summary = {
    "experiment_name": FL_CONFIG["experiment_name"],
    "num_clients": FL_CONFIG["num_clients"],
    "num_rounds": FL_CONFIG["num_rounds"],
    "local_epochs": FL_CONFIG["local_epochs"],
    "base_model": FL_CONFIG["base_model"],
    "best_dev_loss": best_tracker["best_loss"],
    "best_round": best_tracker["best_round"],
    "best_adapter_dir": BEST_ADAPTER_DIR,
    "client_fit_log": CLIENT_FIT_LOG,
    "server_eval_log": DEV_METRICS_LOG,
    "config_path": FL_CONFIG_PATH,
    "split_manifest": manifest_path,
}

summary_path = os.path.join(BASE_PATH, "paper_assets", "tables", "federated_run_summary.json")
save_json(summary_path, summary)
print("Saved run summary to:", summary_path)

Saved run summary to: /content/drive/MyDrive/nirbaan_project/paper_assets/tables/federated_run_summary.json


In [ ]:
from unsloth import FastLanguageModel
import pandas as pd
import os
import torch

def load_best_federated_model():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=FL_CONFIG["lora_r"],
        target_modules=FL_CONFIG["target_modules"],
        lora_alpha=FL_CONFIG["lora_alpha"],
        lora_dropout=FL_CONFIG["lora_dropout"],
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=FL_CONFIG["seed"],
    )

    model.load_adapter(BEST_ADAPTER_DIR, adapter_name="default")
    model.set_adapter("default")

    # Do NOT call FastLanguageModel.for_inference(model) here
    return model, tokenizer


model_best, tokenizer_best = load_best_federated_model()

test_raw = load_jsonl(TEST_PATH)
rows = []

for i, rec in enumerate(test_raw):
    item = normalize_record(rec)
    messages = [{"role": "user", "content": item["prompt"]}]
    text = tokenizer_best.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer_best(
        [text],
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(model_best.device)

    with torch.no_grad():
        outputs = model_best.generate(
            **inputs,
            max_new_tokens=350,
            temperature=0.7,
            top_p=0.95,
            do_sample=True,
            use_cache=False,
        )

    prompt_len = inputs["input_ids"].shape[1]
    generated = tokenizer_best.decode(
        outputs[0][prompt_len:],
        skip_special_tokens=True,
    )

    rows.append({
        "id": i,
        "theme": rec.get("theme", ""),
        "prompt": item["prompt"],
        "reference": item["response"],
        "generated": generated,
    })

test_gen_path = os.path.join(BASE_PATH, "eval", "federated_test_generations.jsonl")
write_jsonl(test_gen_path, rows)
print("Saved test generations to:", test_gen_path)

examples_csv = os.path.join(BASE_PATH, "paper_assets", "examples", "federated_examples.csv")
pd.DataFrame(rows[:10]).to_csv(examples_csv, index=False)
print("Saved example CSV to:", examples_csv)

==((====))==  Unsloth 2026.3.8: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load mlabonne/Meta-Llama-3.1-8B-Instruct-abliterated as a legacy tokenizer.


mlabonne/Meta-Llama-3.1-8B-Instruct-abliterated does not have a padding token! Will use pad_token = <|finetune_right_pad_id|>.


Both `max_new_tokens` (=350) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=350) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=350) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=350) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

Saved test generations to: /content/drive/MyDrive/nirbaan_project/eval/federated_test_generations.jsonl
Saved example CSV to: /content/drive/MyDrive/nirbaan_project/paper_assets/examples/federated_examples.csv


In [32]:
from unsloth import FastLanguageModel
import os

GGUF_DIR = os.path.join(BASE_PATH, "artifacts", "federated_final_gguf")

model_export, tokenizer_export = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model_export = FastLanguageModel.get_peft_model(
    model_export,
    r=FL_CONFIG["lora_r"],
    target_modules=FL_CONFIG["target_modules"],
    lora_alpha=FL_CONFIG["lora_alpha"],
    lora_dropout=FL_CONFIG["lora_dropout"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=FL_CONFIG["seed"],
)

model_export.load_adapter(BEST_ADAPTER_DIR, adapter_name="default")
model_export.set_adapter("default")

model_export.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer_export,
    quantization_method="q4_0",
)

print("Saved GGUF to:", GGUF_DIR)

==((====))==  Unsloth 2026.3.8: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load mlabonne/Meta-Llama-3.1-8B-Instruct-abliterated as a legacy tokenizer.


mlabonne/Meta-Llama-3.1-8B-Instruct-abliterated does not have a padding token! Will use pad_token = <|finetune_right_pad_id|>.
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/drive/MyDrive/nirbaan_project/artifacts/federated_final_gguf_gguf/Meta-Llama-3.1-8B-Instruct-abliterated.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_0. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/drive/MyDrive/nirbaan_project/artifacts/federated_final_gguf_gguf/Meta-Llama-3.1-8B-Instruct-abliterated.Q4_0.gguf']
Unsloth: No Ollama template mapping found for model 'mlabonne/Meta-Llama-3.1-8B-Instruct-abliterated'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /content/drive/MyDrive/nirbaan_project/artifacts/federated_final_gguf_gguf/Meta-Llama-3.1-8B-Instruct-abliterated.Q4_0.gguf -p "why is the sky blue?"
Saved GGUF to: /content/drive/MyDrive/nirbaan_project/artifacts/federated